In [18]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Устройство: {device}")

✅ Устройство: cuda


In [19]:
# ==============================================================================
# МОДУЛЬ 4: Стабильный 2-канальный Guided Restormer для PyTorch 2.0.1
# ==============================================================================
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var = x.var(dim=1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias


class MDTA(nn.Module):
    """Multi-Dconv Head Transposed Attention (Канальное внимание)"""
    def __init__(self, channels, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(1, num_heads, 1, 1))
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=False)
        self.qkv_dwconv = nn.Conv2d(channels * 3, channels * 3, kernel_size=3, padding=1, groups=channels * 3, bias=False)
        self.project_out = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)

        q = q.reshape(b, self.num_heads, -1, h * w)
        k = k.reshape(b, self.num_heads, -1, h * w)
        v = v.reshape(b, self.num_heads, -1, h * w)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.temperature, dim=-1)
        out = torch.matmul(attn, v).reshape(b, c, h, w)
        return self.project_out(out)


class GDFN(nn.Module):
    """Gated-Dconv Feed-Forward Network"""
    def __init__(self, channels, expansion_factor=2.66):
        super().__init__()
        hidden_channels = int(channels * expansion_factor)
        self.project_in = nn.Conv2d(channels, hidden_channels * 2, kernel_size=1, bias=False)
        self.dwconv = nn.Conv2d(hidden_channels * 2, hidden_channels * 2, kernel_size=3, padding=1, groups=hidden_channels * 2, bias=False)
        self.project_out = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        x1, x2 = self.dwconv(self.project_in(x)).chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)


class TransformerBlock(nn.Module):
    def __init__(self, channels, num_heads):
        super().__init__()
        self.norm1 = LayerNorm2d(channels)
        self.attn = MDTA(channels, num_heads)
        self.norm2 = LayerNorm2d(channels)
        self.ffn = GDFN(channels)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class Downsample(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * 2, kernel_size=2, stride=2, bias=False)
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2, bias=False)
    def forward(self, x):
        return self.conv(x)


class RestormerCT(nn.Module):
    """
    Guided Restormer (in_channels=2: Noisy + Canny, out_channels=1: Clean)
    """
    def __init__(self, in_channels=2, out_channels=1, dim=48, num_blocks=[2, 3, 3, 4], num_heads=[1, 2, 4, 8]):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, dim, kernel_size=3, padding=1, bias=False)

        # Энкодер
        self.encoder1 = nn.Sequential(*[TransformerBlock(dim, num_heads[0]) for _ in range(num_blocks[0])])
        self.down1 = Downsample(dim)

        self.encoder2 = nn.Sequential(*[TransformerBlock(dim*2, num_heads[1]) for _ in range(num_blocks[1])])
        self.down2 = Downsample(dim*2)

        self.encoder3 = nn.Sequential(*[TransformerBlock(dim*4, num_heads[2]) for _ in range(num_blocks[2])])
        self.down3 = Downsample(dim*4)

        # Горлышко
        self.bottleneck = nn.Sequential(*[TransformerBlock(dim*8, num_heads[3]) for _ in range(num_blocks[3])])

        # Декодер
        self.up3 = Upsample(dim*8)
        self.reduce3 = nn.Conv2d(dim*8, dim*4, kernel_size=1, bias=False)
        self.decoder3 = nn.Sequential(*[TransformerBlock(dim*4, num_heads[2]) for _ in range(num_blocks[2])])

        self.up2 = Upsample(dim*4)
        self.reduce2 = nn.Conv2d(dim*4, dim*2, kernel_size=1, bias=False)
        self.decoder2 = nn.Sequential(*[TransformerBlock(dim*2, num_heads[1]) for _ in range(num_blocks[1])])

        self.up1 = Upsample(dim*2)
        self.reduce1 = nn.Conv2d(dim*2, dim, kernel_size=1, bias=False)
        self.decoder1 = nn.Sequential(*[TransformerBlock(dim, num_heads[0]) for _ in range(num_blocks[0])])

        # Выходной слой
        self.out_conv = nn.Conv2d(dim, out_channels, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        orig_noisy = x[:, 0:1, :, :]  # Выделяем канал Noisy
        feat = self.in_conv(x)

        f1 = self.encoder1(feat)
        f2 = self.encoder2(self.down1(f1))
        f3 = self.encoder3(self.down2(f2))

        b = self.bottleneck(self.down3(f3))

        d3 = self.decoder3(self.reduce3(torch.cat([self.up3(b), f3], dim=1)))
        d2 = self.decoder2(self.reduce2(torch.cat([self.up2(d3), f2], dim=1)))
        d1 = self.decoder1(self.reduce1(torch.cat([self.up1(d2), f1], dim=1)))

        # Global Residual: Чистый = Шумный МИНУС Предсказанный шум
        predicted_noise = self.out_conv(d1)
        cleaned = orig_noisy - predicted_noise
        return cleaned

In [20]:
model = RestormerCT(in_channels=2, out_channels=1, dim=48).to(device)
model.load_state_dict(torch.load("checkpoints/restormer_best_model.pth", map_location=device))
model.eval()
print("✅ Модель загружена и готова к инференсу")

✅ Модель загружена и готова к инференсу


In [21]:
selected_files = [
    "rec_00416.tif", "rec_00461.tif", "rec_00466.tif", "rec_00476.tif",
    "rec_00481.tif", "rec_00621.tif", "rec_00636.tif", "rec_00661.tif",
    "rec_00686.tif", "rec_00756.tif", "rec_00796.tif", "rec_00806.tif",
    "rec_00906.tif", "rec_00956.tif", "rec_00961.tif", "rec_00971.tif",
    "rec_00996.tif", "rec_01026.tif", "rec_01101.tif", "rec_01111.tif",
    "rec_01271.tif", "rec_01471.tif", "rec_01481.tif", "rec_01546.tif",
    "rec_01691.tif", "rec_01796.tif", "rec_01836.tif", "rec_01906.tif",
    "rec_01911.tif", "rec_01941.tif", "rec_02036.tif", "rec_02061.tif",
    "rec_02131.tif", "rec_02186.tif", "rec_02196.tif", "rec_02231.tif",
    "rec_02256.tif", "rec_02286.tif", "rec_02296.tif", "rec_02351.tif",
    "rec_02366.tif", "rec_02411.tif",
]

In [22]:
NOISY_DIR = "filestore/filestorage/V_beton30_angle05"
REF_DIR   = "filestore/filestorage/V_beton30_angle005"
DATA_RANGE = 65535

CHECKPOINT_PATH = "checkpoints/restormer_best_model.pth"

In [23]:
import tifffile

records = []
for fname in selected_files:
    noisy_path = os.path.join(NOISY_DIR, fname)
    ref_path   = os.path.join(REF_DIR, fname)

    if not (os.path.exists(noisy_path) and os.path.exists(ref_path)):
        print(f"⚠ пропущен, файл не найден: {fname}")
        continue

    noisy = tifffile.imread(noisy_path).astype(np.float32)
    ref   = tifffile.imread(ref_path).astype(np.float32)
    records.append({"name": fname, "noisy": noisy, "ref": ref})

print(f"Загружено {len(records)} из {len(selected_files)} файлов")

Загружено 42 из 42 файлов


In [24]:
def run_model_on_full_image(model, image, device, patch_size=512, overlap_ratio=0.2):
    raise NotImplementedError("Подставьте вашу функцию патчинга+сборки предсказания")

In [25]:
4

4

In [26]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

In [27]:
import pandas as pd

In [28]:
def compute_metrics(pred, ref, data_range):
    return {
        "psnr": psnr(ref, pred, data_range=data_range),
        "ssim": ssim(ref, pred, data_range=data_range),
    }

results = []
for rec in records:
    denoised = run_model_on_full_image(model, rec["noisy"], device,
                                        patch_size=256, overlap_ratio=0.2)
    m = compute_metrics(denoised, rec["ref"], DATA_RANGE)
    m_noisy = compute_metrics(rec["noisy"], rec["ref"], DATA_RANGE)  # baseline до денойзинга
    results.append({
        "name": rec["name"],
        "psnr_denoised": m["psnr"], "ssim_denoised": m["ssim"],
        "psnr_noisy": m_noisy["psnr"], "ssim_noisy": m_noisy["ssim"],
    })
    rec["denoised"] = denoised  # сохраняем для графика профиля ниже

df = pd.DataFrame(results)
print(df)
print("\nСредние значения:")
print(df[["psnr_denoised", "ssim_denoised", "psnr_noisy", "ssim_noisy"]].describe())

df.to_csv("metrics_results.csv", index=False)

NotImplementedError: Подставьте вашу функцию патчинга+сборки предсказания

In [ ]:
import time

def measure_inference_time(model, image, device, n_warmup=2, n_runs=5,
                            patch_size=256, overlap_ratio=0.2):
    for _ in range(n_warmup):
        _ = run_model_on_full_image(model, image, device, patch_size, overlap_ratio)
    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = run_model_on_full_image(model, image, device, patch_size, overlap_ratio)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

sample_image = records[0]["noisy"]
mean_t, std_t = measure_inference_time(model, sample_image, device)
print(f"Время инференса на снимок ({records[0]['name']}, {sample_image.shape}): "
      f"{mean_t:.3f} ± {std_t:.3f} с")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from ipywidgets import interact, FloatSlider

def get_diameter_line(image, center, radius, angle_deg):
    """Берёт значения вдоль линии-диаметра, проходящей через center под углом angle_deg."""
    angle_rad = np.deg2rad(angle_deg)
    dx, dy = np.cos(angle_rad), np.sin(angle_rad)
    t = np.linspace(-radius, radius, int(2 * radius))
    x = center[1] + t * dx
    y = center[0] + t * dy
    coords = np.vstack([y, x])
    values = map_coordinates(image, coords, order=1, mode='nearest')
    return t, values


def plot_profile_at_angle(angle_deg):
    example = records[0]
    h, w = example["noisy"].shape
    center = (h // 2, w // 2)
    radius = min(h, w) // 2

    t, noisy_vals    = get_diameter_line(example["noisy"], center, radius, angle_deg)
    _, ref_vals      = get_diameter_line(example["ref"], center, radius, angle_deg)
    _, denoised_vals = get_diameter_line(example["denoised"], center, radius, angle_deg)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(example["noisy"], cmap="gray")
    angle_rad = np.deg2rad(angle_deg)
    x0 = center[1] - radius * np.cos(angle_rad)
    y0 = center[0] - radius * np.sin(angle_rad)
    x1 = center[1] + radius * np.cos(angle_rad)
    y1 = center[0] + radius * np.sin(angle_rad)
    axes[0].plot([x0, x1], [y0, y1], color="red", linewidth=1)
    axes[0].set_title(f"{example['name']} — угол {angle_deg:.0f}°")
    axes[0].axis("off")

    axes[1].plot(t, noisy_vals, "k--", label="Шумный", alpha=0.7)
    axes[1].plot(t, ref_vals, "k-", label="Эталон")
    axes[1].plot(t, denoised_vals, "r-", label="Денойзнутый (модель)")
    axes[1].legend()
    axes[1].set_title("Профиль интенсивности")

    plt.tight_layout()
    plt.show()


interact(plot_profile_at_angle, angle_deg=FloatSlider(min=0, max=180, step=1, value=0, description="Угол"));

In [ ]:
import os
import cv2
import torch
import numpy as np
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from tqdm import tqdm

# ==========================================
# 1. Настройка метрик (КРИТИЧЕСКИ ВАЖНО)
# ==========================================
# Указываем data_range=1.0, так как мы приводим данные к диапазону [0, 1]
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to('cuda')
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to('cuda')

def rgb_to_ycbcr_torch(img_tensor):
    """
    Конвертирует тензор RGB в YCbCr и возвращает только канал Y (яркость).
    В бенчмарках (SIDD, DND и др.) метрики считаются именно по каналу Y.
    """
    # Коэффициенты для конвертации RGB -> Y
    # Y = 0.299*R + 0.587*G + 0.114*B
    weights = torch.tensor([0.299, 0.587, 0.114], device=img_tensor.device).view(3, 1, 1)
    # img_tensor shape: (C, H, W) -> (1, C, H, W)
    img_tensor = img_tensor.unsqueeze(0) 
    y_channel = torch.sum(img_tensor * weights, dim=1, keepdim=True)
    return y_channel.squeeze(0) # Возвращаем shape (1, H, W)

def load_and_preprocess_image(img_path):
    """
    Загружает изображение, переводит в RGB, нормализует в [0, 1] 
    и конвертирует в float32 тензор.
    """
    # Читаем через OpenCV (вернет BGR)
    img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {img_path}")
    
    # Переводим в RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Если изображение 16-битное (uint16), нормализуем в [0, 1]
    if img_rgb.dtype == np.uint16:
        img_rgb = img_rgb.astype(np.float32) / 65535.0
    elif img_rgb.dtype == np.uint8:
        img_rgb = img_rgb.astype(np.float32) / 255.0
        
    # Конвертируем в PyTorch тензор: (H, W, C) -> (C, H, W)
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1)
    
    return img_tensor

def evaluate_folder(gt_dir, pred_dir, use_y_channel=True):
    """
    Проходит по папкам, считает метрики для каждой пары и выводит среднее.
    """
    gt_files = sorted([f for f in os.listdir(gt_dir) if f.endswith(('.png', '.jpg', '.tif', '.bmp'))])
    
    psnr_list = []
    ssim_list = []
    
    for filename in tqdm(gt_files, desc="Evaluating"):
        gt_path = os.path.join(gt_dir, filename)
        pred_path = os.path.join(pred_dir, filename)
        
        if not os.path.exists(pred_path):
            print(f"Warning: Предсказание для {filename} не найдено.")
            continue
            
        # 1. Загрузка и приведение к [0, 1]
        gt_tensor = load_and_preprocess_image(gt_path).to('cuda')
        pred_tensor = load_and_preprocess_image(pred_path).to('cuda')
        
        # 2. ОБЯЗАТЕЛЬНО: Обрезаем значения, если модель выдала что-то за пределами [0, 1]
        # (например, -0.001 или 1.05). Без этого PSNR может упасть до 8 дБ!
        pred_tensor = torch.clamp(pred_tensor, 0.0, 1.0)
        
        # 3. Перевод в канал Y (если требуется)
        if use_y_channel:
            gt_tensor = rgb_to_ycbcr_torch(gt_tensor)
            pred_tensor = rgb_to_ycbcr_torch(pred_tensor)
            
        # 4. Добавляем размерность батча (B, C, H, W), так как torchmetrics требует этого
        gt_batch = gt_tensor.unsqueeze(0)
        pred_batch = pred_tensor.unsqueeze(0)
        
        # 5. Расчет метрик
        # Сбрасываем состояние метрики для каждого изображения, чтобы считать индивидуально
        psnr_metric.reset()
        ssim_metric.reset()
        
        psnr_val = psnr_metric(pred_batch, gt_batch).item()
        ssim_val = ssim_metric(pred_batch, gt_batch).item()
        
        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        
    # 6. Итоговые средние значения
    avg_psnr = np.mean(psnr_list)
    avg_ssim = np.mean(ssim_list)
    
    print(f"\n--- Результаты ---")
    print(f"Средний PSNR: {avg_psnr:.4f} dB")
    print(f"Средний SSIM: {avg_ssim:.4f}")\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\\
    
    return avg_psnr, avg_ssim

# ==========================================
# Запуск
# ==========================================
if __name__ == "__main__":
    # Укажите пути к папкам с оригиналами (GT) и результатами работы Restormer
    GT_DIRECTORY = "./path/to/ground_truth"
    PRED_DIRECTORY = "./path/to/restormer_output"
    
    # use_y_channel=True рекомендуется для сравнения с официальными бенчмарками
    evaluate_folder(GT_DIRECTORY, PRED_DIRECTORY, use_y_channel=True)

In [ ]:
import cv2

# =========================================================
# Функция нарезки
# =========================================================
def slice_image_to_patches(image_or_path, patch_size=256, overlap_ratio=0.2):
    if isinstance(image_or_path, str):
        img = tifffile.imread(image_or_path).astype(np.float32)
    else:
        img = image_or_path.astype(np.float32)

    h, w = img.shape
    stride = int(patch_size * (1.0 - overlap_ratio))

    y_steps = list(range(0, h - patch_size + 1, stride))
    if y_steps[-1] != h - patch_size: y_steps.append(h - patch_size)

    x_steps = list(range(0, w - patch_size + 1, stride))
    if x_steps[-1] != w - patch_size: x_steps.append(w - patch_size)

    patches, coords = [], []
    for y in y_steps:
        for x in x_steps:
            patches.append(img[y:y+patch_size, x:x+patch_size])
            coords.append((y, x))

    return np.array(patches), coords, (h, w)


# =========================================================
# Функция сборки патчей обратно (взвешенное усреднение, окно Ханна)
# =========================================================
def stitch_patches(pred_patches, coords, orig_shape, patch_size):
    h, w = orig_shape
    canvas = np.zeros((h, w), dtype=np.float32)
    weight = np.zeros((h, w), dtype=np.float32)

    win_1d = np.hanning(patch_size)
    win_1d = np.clip(win_1d, 1e-3, None)
    win_2d = np.outer(win_1d, win_1d)

    for patch, (y, x) in zip(pred_patches, coords):
        canvas[y:y+patch_size, x:x+patch_size] += patch * win_2d
        weight[y:y+patch_size, x:x+patch_size] += win_2d

    weight[weight == 0] = 1.0
    return canvas / weight


# =========================================================
# Главная функция инференса на полном снимке — С ДЕНОРМАЛИЗАЦИЕЙ
# =========================================================
def run_model_on_full_image(model, image, device, patch_size=256, overlap_ratio=0.2):
    noisy_patches, coords, orig_shape = slice_image_to_patches(image, patch_size, overlap_ratio)

    model.eval()
    cleaned_patches = []
    batch_size = 8

    with torch.no_grad():
        for i in range(0, len(noisy_patches), batch_size):
            batch_n = noisy_patches[i:i+batch_size]

            batch_2ch = []
            patch_mins, patch_maxs = [], []
            for p in batch_n:
                p_min, p_max = np.min(p), np.max(p)
                patch_mins.append(p_min)
                patch_maxs.append(p_max)

                p_u8 = ((p - p_min) / (p_max - p_min + 1e-8) * 255.0).astype(np.uint8)
                p_norm = p_u8.astype(np.float32) / 255.0
                canny_u8 = cv2.Canny(cv2.GaussianBlur(p_u8, (3, 3), 1.0), 65, 140)
                canny_norm = canny_u8.astype(np.float32) / 255.0
                tensor_2ch = np.stack([p_norm, canny_norm], axis=0)
                batch_2ch.append(tensor_2ch)

            batch_tensor = torch.from_numpy(np.array(batch_2ch)).float().to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                preds = model(batch_tensor)
            preds_np = preds.squeeze(1).cpu().numpy()

            # ДЕНОРМАЛИЗАЦИЯ: возвращаем выход модели из [0,1] обратно в исходную шкалу патча
            for p_pred, p_min, p_max in zip(preds_np, patch_mins, patch_maxs):
                p_denorm = p_pred * (p_max - p_min + 1e-8) + p_min
                cleaned_patches.append(p_denorm)

    denoised_full = stitch_patches(cleaned_patches, coords, orig_shape, patch_size)
    return denoised_full